# Debug a prediction with TorchCAM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frgfm/notebooks/blob/main/torch-cam/debug_prediction.ipynb)

A classifier can assign a surprising label even when several classes look similar. This notebook compares the predicted class with an **editable expected class**, displays their GradCAM overlays, and saves the results for inspection. The expectation is a hypothesis you supply, not a ground-truth annotation.

We use torchvision's pretrained ResNet-18 and the [PyTorch Hub dog photograph](https://github.com/pytorch/hub/blob/master/images/dog.jpg). The first run downloads the photograph and model weights; later runs reuse local caches. A CPU is sufficient.


## 1. Install a reproducible TorchCAM snapshot

Use **Python 3.11 or newer**. `explain()` and the numerical fixes used here are available in this pinned development snapshot, after the 0.5.0 release. The `[scripts]` extra installs torchvision. Run this cell once; restart the kernel if the notebook interface requests it. The version report below records the runtime packages, whose versions can vary across environments.


In [ ]:
%pip install "torchcam[scripts] @ git+https://github.com/frgfm/torch-cam.git@e311623a60474aeab359afee5c432cc97a27269e"


In [ ]:
import json
import math
import platform
import tempfile
from importlib.metadata import version
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms.functional as TF
from IPython.display import Markdown, display
from PIL import Image
from torchvision.models import ResNet18_Weights, resnet18

from torchcam.explain import explain
from torchcam.utils import overlay_mask

SOURCE_REVISION = "e311623a60474aeab359afee5c432cc97a27269e"
# Keep CPU execution light; a CUDA runtime uses the same notebook cells.
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
runtime_versions = {
    "python": platform.python_version(),
    **{package: version(package) for package in ("torch", "torchvision", "torchcam", "Pillow", "matplotlib")},
}
print(json.dumps({"source_revision": SOURCE_REVISION, "device": str(device), **runtime_versions}, indent=2))


## 2. Load the image and match the model's coordinate frame

Use the weight enum's preprocessing for inference. It resizes the photograph and takes a center crop before normalization. **The displayed and saved overlays must use that same crop**: stretching a crop-based CAM over the original photograph would put the highlighted regions in the wrong places.

The check below reconstructs the normalized input from the displayed crop and verifies that it matches `weights.transforms()` exactly. To use your own photo, replace the cached image path and keep this alignment step.


In [ ]:
IMAGE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
cache_dir = Path.home() / ".cache" / "torchcam-notebooks"
cache_dir.mkdir(parents=True, exist_ok=True)
image_path = cache_dir / "dog.jpg"
if not image_path.is_file():
    with urlopen(IMAGE_URL, timeout=30) as response:
        image_bytes = response.read()
    # Decode before caching so an incomplete download is not kept as the sample.
    with Image.open(BytesIO(image_bytes)) as downloaded:
        downloaded.verify()
    image_path.write_bytes(image_bytes)
with Image.open(image_path) as image_file:
    original_image = image_file.convert("RGB")

weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
class_names = weights.meta["categories"]
input_tensor = preprocess(original_image).unsqueeze(0).to(device)

aligned_image = TF.center_crop(
    TF.resize(
        original_image,
        preprocess.resize_size,
        interpolation=preprocess.interpolation,
        antialias=preprocess.antialias,
    ),
    preprocess.crop_size,
)
aligned_tensor = TF.normalize(TF.to_tensor(aligned_image), preprocess.mean, preprocess.std)
assert torch.equal(aligned_tensor, input_tensor[0].cpu())
assert aligned_image.size == (input_tensor.shape[-1], input_tensor.shape[-2])

model = resnet18(weights=weights).to(device).eval()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, image, title in zip(
    axes, (original_image, aligned_image), ("Original photograph", "Exact model input crop"), strict=True
):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()


## 3. Supply an expected class and explain both scores

For the comparison, try **Great Pyrenees**, an ImageNet class for another white dog breed. This is an example hypothesis, not a claim about the photograph's correct label. Choose another exact name from `class_names`, or set the expectation to `None` to explain only the prediction.

`explain()` defaults to GradCAM. We name ResNet's `layer4` explicitly and keep the model in evaluation mode. Run the explanation outside `torch.inference_mode()` because CAM extraction needs gradients. When the prediction and expectation are the same class, the helper returns one shared map and this notebook labels it with both roles.


In [ ]:
expected_class_name = "Great Pyrenees"  # Editable hypothesis; use None for prediction only.
name_to_index = {name: index for index, name in enumerate(class_names)}
if expected_class_name is not None and expected_class_name not in name_to_index:
    raise ValueError("Choose an exact ImageNet category from class_names, or use None.")
expected_class_idx = None if expected_class_name is None else name_to_index[expected_class_name]

result = explain(
    model,
    input_tensor,
    expected_class_idx=expected_class_idx,
    class_names=class_names,
    target_layer="layer4",
)
probabilities = result.logits.softmax(dim=1)[0]
assert result.logits.shape == (1, len(class_names))
assert torch.isfinite(result.logits).all()
assert torch.isfinite(probabilities).all()
assert torch.all((probabilities >= 0) & (probabilities <= 1))
assert math.isclose(float(probabilities.sum()), 1.0, abs_tol=1e-6)
assert result.method == "GradCAM"
assert result.target_layers == ("layer4",)

print(f"Prediction: {class_names[result.predicted_class_idx]} ({probabilities[result.predicted_class_idx]:.2%})")
if expected_class_idx is not None:
    print(f"Expectation: {class_names[expected_class_idx]} ({probabilities[expected_class_idx]:.2%})")
    if result.predicted_class_idx == expected_class_idx:
        print("Prediction and expectation match; one shared CAM is sufficient.")

top_k = min(5, probabilities.numel())
top_probabilities, top_indices = probabilities.topk(top_k)
rows = ["| Class | Probability |", "|---|---:|"]
rows += [
    f"| {class_names[index]} | {probability:.2%} |"
    for index, probability in zip(top_indices.tolist(), top_probabilities.tolist(), strict=True)
]
display(Markdown(f"Top {top_k} predictions:\n\n" + "\n".join(rows)))


In [ ]:
selected_classes = list(dict.fromkeys(
    [result.predicted_class_idx] + ([] if expected_class_idx is None else [expected_class_idx])
))
fig, axes = plt.subplots(1, 1 + len(selected_classes), figsize=(5 * (1 + len(selected_classes)), 4))
axes[0].imshow(aligned_image)
axes[0].set_title("Model input crop")
axes[0].axis("off")
for axis, class_idx in zip(axes[1:], selected_classes, strict=True):
    maps = result.cams[class_idx]
    assert len(maps) == 1  # One explicitly selected target layer.
    cam = maps[0]
    assert cam.ndim == 2 and torch.isfinite(cam).all()
    roles = []
    if class_idx == result.predicted_class_idx:
        roles.append("predicted")
    if class_idx == expected_class_idx:
        roles.append("expected")
    overlay = overlay_mask(aligned_image, Image.fromarray(cam.numpy()), alpha=0.5)
    axis.imshow(overlay)
    axis.set_title(f"{' + '.join(roles)}: {class_names[class_idx]}\n{probabilities[class_idx]:.2%}")
    axis.axis("off")
plt.tight_layout()
plt.show()


The probability table tells you how the classifier ranks labels. The overlays help you inspect the regions associated with each selected class score and formulate follow-up checks, such as testing a tighter crop or collecting examples of similar breeds.

GradCAM is coarse at `layer4`, and each map is normalized independently. A brighter overlay is **not** evidence that one class has a higher probability, that the region caused the prediction, or that the prediction is correct. Softmax probabilities are not automatically calibrated. If prediction and expectation agree, the shared CAM still needs the same careful interpretation.


## 4. Save an evidence bundle and inspect its manifest

`result.save()` creates a **new** directory and refuses to overwrite an existing one. Each run below creates a unique parent directory, so rerunning the cell is safe. The schema-v1 manifest records the selected classes, probabilities, logits, method, layers, input shape, versions, and filenames. NumPy arrays retain the numerical maps; PNGs provide heatmaps and overlays.

We also save `input-crop.png` and `notebook-context.json` alongside those artifacts to preserve the exact displayed image, weight selection, source revision, and preprocessing. These two notebook-added files are separate from the library's manifest schema. Download the bundle before a temporary notebook runtime ends.


In [ ]:
bundle_parent = Path(tempfile.mkdtemp(prefix="torchcam-prediction-"))
bundle_dir = result.save(bundle_parent / "evidence", aligned_image, alpha=0.5)
aligned_image.save(bundle_dir / "input-crop.png")
notebook_context = {
    "torchcam_source_revision": SOURCE_REVISION,
    "weights": "ResNet18_Weights.IMAGENET1K_V1",
    "image_url": IMAGE_URL,
    "input_image": "input-crop.png",
    "expected_class_is": "user-supplied hypothesis, not a ground-truth annotation",
    "preprocessing": {
        "resize_size": preprocess.resize_size,
        "crop_size": preprocess.crop_size,
        "interpolation": str(preprocess.interpolation),
        "antialias": preprocess.antialias,
        "mean": preprocess.mean,
        "std": preprocess.std,
    },
    "runtime_versions": runtime_versions,
}
(bundle_dir / "notebook-context.json").write_text(
    json.dumps(notebook_context, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
manifest = json.loads((bundle_dir / "manifest.json").read_text(encoding="utf-8"))
print(f"Saved evidence to: {bundle_dir}")
print(json.dumps(manifest, indent=2))


In [ ]:
def class_reference(class_idx):
    return None if class_idx is None else {
        "class_idx": class_idx, "class_name": class_names[class_idx]
    }

assert manifest["schema_version"] == 1
assert manifest["prediction"] == class_reference(result.predicted_class_idx)
assert manifest["expected"] == class_reference(expected_class_idx)
assert manifest["method"] == result.method
assert manifest["target_layers"] == list(result.target_layers)
assert manifest["model"] == result.model
assert manifest["input_shape"] == list(result.input_shape)
assert manifest["image_size"] == list(aligned_image.size)
assert manifest["versions"] == dict(result.versions)
assert manifest["alpha"] == 0.5
assert set(manifest["classes"]) == {str(index) for index in selected_classes}

for class_idx in selected_classes:
    entry = manifest["classes"][str(class_idx)]
    assert entry["class_idx"] == class_idx
    assert entry["class_name"] == class_names[class_idx]
    assert math.isfinite(entry["logit"]) and math.isfinite(entry["probability"])
    assert 0 <= entry["probability"] <= 1
    assert math.isclose(entry["logit"], float(result.logits[0, class_idx]), abs_tol=1e-7)
    assert math.isclose(entry["probability"], float(probabilities[class_idx]), abs_tol=1e-7)
    assert len(entry["artifacts"]) == len(result.cams[class_idx])
    for artifact, cam in zip(entry["artifacts"], result.cams[class_idx], strict=True):
        assert artifact["target_layers"] == list(result.target_layers)
        for key in ("map", "heatmap", "overlay"):
            # The save API emits local basenames, not paths outside the bundle.
            assert Path(artifact[key]).name == artifact[key]
            assert (bundle_dir / artifact[key]).is_file()
        saved_map = np.load(bundle_dir / artifact["map"], allow_pickle=False)
        assert saved_map.ndim == 2 and saved_map.shape == tuple(cam.shape)
        assert saved_map.dtype == np.float32 and np.isfinite(saved_map).all()
        assert np.all((saved_map >= 0) & (saved_map <= 1))
        np.testing.assert_array_equal(saved_map, cam.numpy())
        with Image.open(bundle_dir / artifact["heatmap"]) as heatmap:
            assert heatmap.size == (saved_map.shape[1], saved_map.shape[0])
            assert heatmap.mode == "L"
        with Image.open(bundle_dir / artifact["overlay"]) as overlay:
            assert overlay.size == aligned_image.size
            assert np.isfinite(np.asarray(overlay)).all()
with Image.open(bundle_dir / "input-crop.png") as saved_input:
    np.testing.assert_array_equal(np.asarray(saved_input), np.asarray(aligned_image))
assert json.loads((bundle_dir / "notebook-context.json").read_text(encoding="utf-8")) == notebook_context
print(f"Verified schema-v1 manifest and all {sum(len(entry['artifacts']) for entry in manifest['classes'].values())} CAM artifact sets.")


## Adapt this workflow to your own classifier

Replace the model and weights in section 2, along with the matching preprocessing, class names, and expected-class lookup. Select a target layer suitable for that architecture; `layer4` is specific to this ResNet example. Preserve the resize/crop coordinate alignment when displaying and saving your own image.

The high-level `explain()` helper currently handles **one 2D image**: a floating-point tensor shaped `(1, channels, height, width)` and an evaluation-mode model returning finite logits shaped `(1, num_classes)`. Wrap a model that returns a dictionary or tuple so that its forward returns the desired logits tensor. Keep label indices aligned with the model's output order and provide raw logits rather than softmax probabilities. Segmentation, batched inputs, and video need task-specific targets and the lower-level extractor APIs.

For a prediction without an expected class, set `expected_class_name = None` and rerun sections 3–4. To inspect a matching expectation, set it to the displayed predicted category. Both cases use the same plotting and evidence-saving code.
